# Robustness Experiments
This notebook reproduces the epsilon-robustness experiments found in the paper "Towards Robust Classification with Deep Generative Forests".

In [1]:
import numpy as np
from tqdm import tqdm

from load_data import load_dataset

from visuals.accrej_plots import get_mean_accs, get_accaccs_curve, get_mean_accs_hybrid
from visuals.store_results import write_results_to_csv

from experiments_NBC import standard_experiment_on_train_NBC
from experiments_GeFs import standard_experiment_on_train_GeFs

This is the notebook reproduces the experiments for the paper "Robustness Quantification and Uncertainty Quantification: Comparing Two Methods for Assessing the Reliability of Classifier Predictions" (2026).
The experiments won't be exactly the same as in the paper, because the code was refactored and some important implementations were changed. However, the results should be similar.

In [2]:
def get_mean_accs_new_hybrid(list_of_rob_metrics, list_of_classif_results, metric_1, metric_2, mixing, stddev=False):
    list_of_accs = []
    list_of_aucs = []

    for i in range(len(list_of_rob_metrics)):
        hybrid = mixing * list_of_rob_metrics[i][metric_1] + (1-mixing) * list_of_rob_metrics[i][metric_2]

        accs = get_accaccs_curve(hybrid, list_of_classif_results[i])
        list_of_accs.append(accs)
        list_of_aucs.append(np.mean(accs))

    list_of_accs = np.vstack(list_of_accs)
    list_of_aucs = np.vstack(list_of_aucs)

    accaccs = np.mean(list_of_accs, axis=0)
    auc = np.mean(list_of_aucs)

    if stddev:
        stddevs = np.std(list_of_accs, axis=0)
        stddevs_auc = np.std(list_of_aucs)
        return accaccs, auc, stddevs, stddevs_auc
    return accaccs, auc

def find_best_hybrid_metric(list_of_rob_metrics, list_of_rob_metrics_train, list_of_classif_results, list_of_classif_results_train, metric_1, metric_2, stddev=True, hybrid_to_compute=None):
	best_mixing = 0.0
	best_auc = -100
	for mixing in np.linspace(0.0, 1.0, num=101):
		if hybrid_to_compute == 'new':
			_, auc_hybrid_train = get_mean_accs_new_hybrid(list_of_rob_metrics_train, list_of_classif_results_train, metric_1=metric_1, metric_2=metric_2, mixing=mixing, stddev=False)
		else:
			_, auc_hybrid_train = get_mean_accs_hybrid(list_of_rob_metrics_train, list_of_classif_results_train, metric_1, metric_2=metric_2, mixing=mixing)
		if auc_hybrid_train > best_auc:
			best_auc = auc_hybrid_train
			best_mixing = mixing

	if stddev:
		if hybrid_to_compute == 'new':
			hybrid_accs, auc_hybrid, hybrid_stddevs, stddevs_auc_hybrid = get_mean_accs_new_hybrid(list_of_rob_metrics, list_of_classif_results, metric_1=metric_1, metric_2=metric_2, mixing=best_mixing, stddev=True)
		else:
			hybrid_accs, auc_hybrid, hybrid_stddevs, stddevs_auc_hybrid = get_mean_accs_hybrid(list_of_rob_metrics, list_of_classif_results, metric_1, metric_2=metric_2, mixing=best_mixing, stddev=True)

		return best_mixing, hybrid_accs, auc_hybrid, hybrid_stddevs, stddevs_auc_hybrid
	else:
		if hybrid_to_compute == 'new':
			hybrid_accs, auc_hybrid = get_mean_accs_new_hybrid(list_of_rob_metrics, list_of_classif_results, metric_1=metric_1, metric_2=metric_2, mixing=best_mixing, stddev=False)
		else:
			hybrid_accs, auc_hybrid = get_mean_accs_hybrid(list_of_rob_metrics, list_of_classif_results, metric_1, metric_2=metric_2, mixing=best_mixing, stddev=False)
		return best_mixing, hybrid_accs, auc_hybrid



def find_optimal_metric(list_of_rob_metrics, list_of_classif_results, metric_1, metric_2, hybrid_to_compute=None):
	opt_mixing = 0.0
	opt_auc = -100
	for mixing in np.linspace(0.0, 1.0, num=101):
		if hybrid_to_compute == 'new':
			_, auc_hybrid_train = get_mean_accs_new_hybrid(list_of_rob_metrics, list_of_classif_results, metric_1=metric_1, metric_2=metric_2, mixing=mixing, stddev=False)
		else:
			_, auc_hybrid_train = get_mean_accs_hybrid(list_of_rob_metrics, list_of_classif_results, metric_1, metric_2=metric_2, mixing=mixing)
		if auc_hybrid_train > opt_auc:
			opt_auc = auc_hybrid_train
			opt_mixing = mixing

	return opt_mixing, opt_auc


def normalize_metrics(list_of_rob_metrics, metrics_to_compute):
	reverse_order_metrics = ['entropies', 'aleatoric', 'epistemic', 'total']

	list_of_rob_metrics_norm = [rob_metric.copy() for rob_metric in list_of_rob_metrics]

	for metric in metrics_to_compute:
		for dict_index in range(len(list_of_rob_metrics)):
			rob_metric_cp = list_of_rob_metrics_norm[dict_index][metric].copy()
			if metric in reverse_order_metrics:
				rob_metric_cp *= -1
			R = (rob_metric_cp - np.min(rob_metric_cp)) / (np.max(rob_metric_cp) - np.min(rob_metric_cp))
			if metric in reverse_order_metrics:
				R = 1 - R
			list_of_rob_metrics_norm[dict_index][metric] = R

	return list_of_rob_metrics_norm


# def make_numeric(X, feat_vals):
# 	X_num = np.zeros(X.shape)
# 	for i in range(len(feat_vals)):
# 		X_num[:, i] = np.searchsorted(feat_vals[i], X[:, i])
# 	X_num = X_num.astype(int)
# 	return X_num

In [3]:
def introduce_label_noise(y_train, number_of_classes, noise_fraction=0.05, random_state=42):
	np.random.seed(random_state)
	n_samples = len(y_train)
	n_noisy = int(noise_fraction * n_samples)
	noisy_indices = np.random.choice(n_samples, n_noisy, replace=False)

	y_train_noisy = y_train.copy()
	for index in noisy_indices:
		current_label = y_train[index]
		possible_labels = np.arange(number_of_classes)
		possible_labels = possible_labels[possible_labels != current_label]
		new_label = np.random.choice(possible_labels)
		y_train_noisy[index] = new_label

	return y_train_noisy

In [ ]:
seed = 0

np.random.seed(seed)  # for reproducibility
print(f"\n--------------- Running experiment with seed: {seed} ---------------")

store_results = True  # whether to store the results in a csv/txt files

# For GeFs
n_estimators = 30
precision_bisearch_GeFs = 1e-9
max_iter_GeFs = 100


n_splits = 5 # number of split for cross-validation in NBC
N_bootstraps = 10 # number of bootstrap models for calculating the information-theoretic uncertainty metrics


noise_factors = [0.0, 0.10, 0.20]

n_training_sessions = 3
n_runs_per_session = 7
multiple_runs = [n_runs_per_session] * (n_training_sessions)
# add None if you want to run the experiments on the full training set as well
max_n_train_sets = [50, 100, 200]



fraction_test = 0.4  # fraction of the data to use as a test set if no test set is provided
max_size_test_set = 1000  # maximum size of the test set if no test set is provided


sets_for_experiments = ['Australian',
		'adult',
		'bank',
		'BCW',
		'german',
		'Healthy_aging',
		'heart_disease',
		'lymphography',
		'nursery',
		'solar_big',
		'solar_small',
		'spect_heart',
		'student_math_bin',
		'student_port_bin']


for training_session in range(n_training_sessions):
	n_train_runs = multiple_runs[training_session]
	max_size_train_set = max_n_train_sets[training_session]

	if max_size_train_set is not None:
		limited_data = True
		multiple_train_runs = True
		train_N_suffix = f'_{max_size_train_set}'
	else:
		limited_data = False
		multiple_train_runs = False
		train_N_suffix = f'_ALL'


	for set_name in sets_for_experiments:
		print(f"\n ------- {set_name} -------\n")

		# name to use when storing the results
		name = f"{set_name}_{seed}" + train_N_suffix


		##### Choose reliability metric(s) to compute/compare and some additional parameters ######
		uncertainty_metrics = ['max_prob', 'entropies', 'aleatoric', 'epistemic', 'total', 'margin_of_confidence']
		robustness_metrics = ['global_eps', 'local_eps']

		metrics_to_compute = uncertainty_metrics + robustness_metrics

		rob_metrics_shifted_NBC = {}
		rob_metrics_shifted_GeFs = {}
		for metric in metrics_to_compute:
			rob_metrics_shifted_NBC[metric] = []
			rob_metrics_shifted_GeFs[metric] = []

		for noise_factor in noise_factors:
			name = f"{set_name}_{seed}" + train_N_suffix + f"_noise_{noise_factor}"

			if noise_factor > 0:
				label_noise = True
			else:
				label_noise = False

			##### Run experiments ######
			list_of_rob_metrics_NBC = []
			list_of_classif_results_NBC = []
			list_of_rob_metrics_GeFs = []
			list_of_classif_results_GeFs = []

			list_of_rob_metrics_train_NBC = []
			list_of_classif_results_train_NBC = []
			list_of_rob_metrics_train_GeFs = []
			list_of_classif_results_train_GeFs = []


			for _ in tqdm(range(n_train_runs)):

				if set_name in sets_for_experiments:
					X, y, X_test, y_test, C, F = load_dataset(set_name, randomize=True)
					X_train = None
				else:
					print("UNKNOWN DATASET")
					continue


				# if no test set is provided, split the data into a test and train set
				if X_test is None:
					test_set_size = int(fraction_test * X.shape[0])

					if test_set_size > max_size_test_set:
						test_set_size = max_size_test_set

					# random ordering of the data
					perm = np.random.permutation(len(y))
					X, y = X[perm], y[perm]

					X_test = X[:test_set_size]
					y_test = y[:test_set_size]

					X_train = X[test_set_size:]
					y_train = y[test_set_size:]

				elif X_train is None:
					X_train = X
					y_train = y

					if X_test.shape[0] > max_size_test_set:
						X_test = X_test[:max_size_test_set]
						y_test = y_test[:max_size_test_set]
				else:
					X_test = X_test[:max_size_test_set]
					y_test = y_test[:max_size_test_set]


				# Introduce label noise
				if label_noise:
					y_train_noisy = introduce_label_noise(y_train, C, noise_fraction=noise_factor)


				if limited_data:
					# shuffle and take a limited number of data
					perm = np.random.permutation(len(y_train))
					if label_noise:
						X_t = X_train[perm][:max_size_train_set]
						y_t = y_train_noisy[perm][:max_size_train_set]
					else:
						X_t = X_train[perm][:max_size_train_set]
						y_t = y_train[perm][:max_size_train_set]
				else:
					if label_noise:
						X_t = X_train
						y_t = y_train_noisy
					else:
						X_t = X_train
						y_t = y_train


				### NBC ###
				rob_metrics, classification_results, y_pred, rob_metrics_train, classification_results_train, y_pred_train = standard_experiment_on_train_NBC(X_t, y_t, X_test, y_test, C, F, metrics_to_compute, n_splits=n_splits, N_bootstraps=N_bootstraps)

				list_of_rob_metrics_NBC.append(rob_metrics)
				list_of_classif_results_NBC.append(classification_results)

				list_of_rob_metrics_train_NBC.append(rob_metrics_train)
				list_of_classif_results_train_NBC.append(classification_results_train)


				### GeFs ###
				rob_metrics, classification_results, y_pred, rob_metrics_train, classification_results_train, y_pred_train = standard_experiment_on_train_GeFs(X_t, y_t, X_test, y_test, C, F, metrics_to_compute, n_estimators=n_estimators, N_bootstraps=N_bootstraps, precision_bisearch=precision_bisearch_GeFs, max_iter=max_iter_GeFs)

				list_of_rob_metrics_GeFs.append(rob_metrics)
				list_of_classif_results_GeFs.append(classification_results)

				list_of_rob_metrics_train_GeFs.append(rob_metrics_train)
				list_of_classif_results_train_GeFs.append(classification_results_train)


			list_of_rob_classif_results_NBC = np.vstack(list_of_classif_results_NBC)
			list_of_rob_classif_results_train_NBC = np.vstack(list_of_classif_results_train_NBC)

			list_of_rob_classif_results_GeFs = np.vstack(list_of_classif_results_GeFs)
			list_of_rob_classif_results_train_GeFs = np.vstack(list_of_classif_results_train_GeFs)


			print('Accuracy NBC: ', np.mean(list_of_classif_results_NBC))
			print('Accuracy GeFs: ', np.mean(list_of_classif_results_GeFs))


			if store_results:
				# generate random number to append to the filename to avoid overwriting
				list_of_rob_metrics_NBC_cp = list_of_rob_metrics_NBC.copy()
				write_results_to_csv(list_of_rob_classif_results_NBC, list_of_rob_metrics_NBC_cp, list_of_robm=True, set_name=name+"_NBC", folder='results_UAI/csv')
				list_of_rob_metrics_GeFs_cp = list_of_rob_metrics_GeFs.copy()
				write_results_to_csv(list_of_rob_classif_results_GeFs, list_of_rob_metrics_GeFs_cp, list_of_robm=True, set_name=name+"_GeFs", folder='results_UAI/csv')


			# transform all metrics to the same scale (0-1) for better comparison in the plots
			list_of_rob_metrics_NBC_norm = normalize_metrics(list_of_rob_metrics_NBC, metrics_to_compute)
			list_of_rob_metrics_train_NBC_norm = normalize_metrics(list_of_rob_metrics_train_NBC, metrics_to_compute)

			list_of_rob_metrics_GeFs_norm = normalize_metrics(list_of_rob_metrics_GeFs, metrics_to_compute)
			list_of_rob_metrics_train_GeFs_norm = normalize_metrics(list_of_rob_metrics_train_GeFs, metrics_to_compute)


			### STEP 5: Plot the robustness-uncertainty point clouds
			metrics_fixed = ['global_eps', 'local_eps']
			metrics = ['margin_of_confidence', 'max_prob', "aleatoric", 'total', 'epistemic', 'entropies']

			better_NBC = []
			better_GeFs = []
			not_better_NBC = []
			not_better_GeFs = []

			results_stored = []
			auc_stored = False

			all_aucs_NBC = []
			opt_aucs_NBC = []
			all_aucs_GeFs = []
			opt_aucs_GeFs = []

			metrics_stored = []

			show_optimal_mixing = True

			for metric_fixed in metrics_fixed:

				accs_fixed_metric_NBC, auc_fixed_metric_NBC, stddevs_fixed_metric_NBC, stddevs_auc_fixed_metric_NBC = get_mean_accs(list_of_rob_metrics_NBC, list_of_classif_results_NBC, metric=metric_fixed, stddev=True)
				all_aucs_NBC.append((auc_fixed_metric_NBC, metric_fixed))
				opt_aucs_NBC.append((auc_fixed_metric_NBC, metric_fixed))

				accs_fixed_metric_GeFs, auc_fixed_metric_GeFs, stddevs_fixed_metric_GeFs, stddevs_auc_fixed_metric_GeFs = get_mean_accs(list_of_rob_metrics_GeFs, list_of_classif_results_GeFs, metric=metric_fixed, stddev=True)
				all_aucs_GeFs.append((auc_fixed_metric_GeFs, metric_fixed))
				opt_aucs_GeFs.append((auc_fixed_metric_GeFs, metric_fixed))

				rob_metrics_shifted_NBC[metric_fixed].append((auc_fixed_metric_NBC, stddevs_auc_fixed_metric_NBC))
				rob_metrics_shifted_GeFs[metric_fixed].append((auc_fixed_metric_GeFs, stddevs_auc_fixed_metric_GeFs))

				if store_results:
					filename = f'results_UAI/accaccs/{name}_NBC.dat'
					with open(filename, 'a') as f:
						f.write(f'fixed:{metric_fixed}\n')
						f.write(','.join([str(x) for x in accs_fixed_metric_NBC]) + '\n')

					filename = f'results_UAI/accaccs/{name}_GeFs.dat'
					with open(filename, 'a') as f:
						f.write(f'fixed:{metric_fixed}\n')
						f.write(','.join([str(x) for x in accs_fixed_metric_GeFs]) + '\n')


				for metric in metrics:
					if metric == metric_fixed:
						continue

					accs_metric_NBC, auc_metric_NBC, stddevs_metric_NBC, stddevs_auc_metric_NBC = get_mean_accs(list_of_rob_metrics_NBC, list_of_classif_results_NBC, metric=metric, stddev=True)
					accs_metric_GeFs, auc_metric_GeFs, stddevs_metric_GeFs, stddevs_auc_metric_GeFs = get_mean_accs(list_of_rob_metrics_GeFs, list_of_classif_results_GeFs, metric=metric, stddev=True)
					if not auc_stored:
						all_aucs_NBC.append((auc_metric_NBC, metric))
						all_aucs_GeFs.append((auc_metric_GeFs, metric))
						opt_aucs_NBC.append((auc_metric_NBC, metric))
						opt_aucs_GeFs.append((auc_metric_GeFs, metric))

						rob_metrics_shifted_NBC[metric].append((auc_metric_NBC, stddevs_auc_metric_NBC))
						rob_metrics_shifted_GeFs[metric].append((auc_metric_GeFs, stddevs_auc_metric_GeFs))



					best_mixing_NBC, hybrid_accs_NBC, auc_hybrid_NBC, hybrid_stddevs_NBC, stddevs_auc_hybrid_NBC = find_best_hybrid_metric(list_of_rob_metrics_NBC_norm, list_of_rob_metrics_train_NBC_norm, list_of_classif_results_NBC, list_of_classif_results_train_NBC, metric_1=metric, metric_2=metric_fixed, stddev=True, hybrid_to_compute='new')
					all_aucs_NBC.append((auc_hybrid_NBC, f'{metric_fixed}_{metric}_{best_mixing_NBC:.2f}'))


					# find optimal mixing on the test set as well
					optimal_mixing_NBC, optimal_auc_NBC = find_optimal_metric(list_of_rob_metrics_NBC_norm, list_of_classif_results_NBC, metric_1=metric, metric_2=metric_fixed)
					opt_aucs_NBC.append((optimal_auc_NBC, f'{metric_fixed}_{metric}_{optimal_mixing_NBC:.2f}'))



					best_mixing_GeFs, hybrid_accs_GeFs, auc_hybrid_GeFs, hybrid_stddevs_GeFs, stddevs_auc_hybrid_GeFs = find_best_hybrid_metric(list_of_rob_metrics_GeFs_norm, list_of_rob_metrics_train_GeFs_norm, list_of_classif_results_GeFs, list_of_classif_results_train_GeFs, metric_1=metric, metric_2=metric_fixed, stddev=True, hybrid_to_compute='new')
					all_aucs_GeFs.append((auc_hybrid_GeFs, f'{metric_fixed}_{metric}_{best_mixing_GeFs:.2f}'))


					# find optimal mixing on the test set as well
					optimal_mixing_GeFs, optimal_auc_GeFs = find_optimal_metric(list_of_rob_metrics_GeFs_norm, list_of_classif_results_GeFs, metric_1=metric, metric_2=metric_fixed)
					opt_aucs_GeFs.append((optimal_auc_GeFs, f'{metric_fixed}_{metric}_{optimal_mixing_GeFs:.2f}'))


					if store_results:
						filename = f'results_UAI/accaccs/{name}_NBC.dat'
						# only store accs_metric the first time
						if metric_fixed == metrics_fixed[0]:

							with open(filename, 'a') as f:
								f.write(f'uncer:{metric}\n')
								f.write(','.join([str(x) for x in accs_metric_NBC]) + '\n')

						with open(filename, 'a') as f:
							f.write(f'hybri:{metric_fixed},{metric}\n')
							f.write(','.join([str(x) for x in hybrid_accs_NBC]) + '\n')


						filename = f'results_UAI/accaccs/{name}_GeFs.dat'
						# only store accs_metric the first time
						if metric_fixed == metrics_fixed[0]:

							with open(filename, 'a') as f:
								f.write(f'uncer:{metric}\n')
								f.write(','.join([str(x) for x in accs_metric_GeFs]) + '\n')

						with open(filename, 'a') as f:
							f.write(f'hybri:{metric_fixed},{metric}\n')
							f.write(','.join([str(x) for x in hybrid_accs_GeFs]) + '\n')


					if store_results:
						delimiter = ','
						filename = f'results_UAI/auc/{name}_NBC.dat'

						with open(filename, 'a') as f:
							# write first line with column names
							if show_optimal_mixing:
								f.write(f'AUC {metric_fixed:7.7} vs {metric:5.5} vs Hybrid {best_mixing_NBC:.2f} ({optimal_mixing_NBC:.2f}): {auc_fixed_metric_NBC:.8f} {auc_metric_NBC:.8f} {auc_hybrid_NBC:.8f} ({optimal_auc_NBC:.8f})\n\n')
							else:
								f.write(f'AUC {metric_fixed:7.7} vs {metric:5.5} vs Hybrid {best_mixing_NBC:.2f}: {auc_fixed_metric_NBC:.8f} {auc_metric_NBC:.8f} {auc_hybrid_NBC:.8f}\n\n')


						delimiter = ','
						filename = f'results_UAI/auc/{name}_GeFs.dat'

						with open(filename, 'a') as f:
							# write first line with column names
							if show_optimal_mixing:
								f.write(f'AUC {metric_fixed:7.7} vs {metric:5.5} vs Hybrid {best_mixing_GeFs:.2f} ({optimal_mixing_GeFs:.2f}): {auc_fixed_metric_GeFs:.8f} {auc_metric_GeFs:.8f} {auc_hybrid_GeFs:.8f} ({optimal_auc_GeFs:.8f})\n\n')
							else:
								f.write(f'AUC {metric_fixed:7.7} vs {metric:5.5} vs Hybrid {best_mixing_GeFs:.2f}: {auc_fixed_metric_GeFs:.8f} {auc_metric_GeFs:.8f} {auc_hybrid_GeFs:.8f}\n\n')


				auc_stored = True

				if store_results:
					delimiter = ','
					filename = f'results_UAI/auc/{name}_NBC.dat'

					with open(filename, 'a') as f:
						f.write('\n')

					delimiter = ','
					filename = f'results_UAI/auc/{name}_GeFs.dat'

					with open(filename, 'a') as f:
						f.write('\n')

			# sort all_aucs descending on the AUC value
			all_aucs_NBC.sort(key=lambda x: x[0], reverse=True)
			opt_aucs_NBC.sort(key=lambda x: x[0], reverse=True)
			all_aucs_GeFs.sort(key=lambda x: x[0], reverse=True)
			opt_aucs_GeFs.sort(key=lambda x: x[0], reverse=True)

			if store_results:
				delimiter = ','
				filename = f'results_UAI/auc/{name}_NBC.dat'

				empty = ' '

				with open(filename, 'a') as f:
					f.write(f'\nRanking of all metrics and combinations by AUC: {empty:44s} Ranking of all metrics and OPTIMAL combinations by AUC:\n\n')
					for index in range(len(all_aucs_NBC)):
						f.write(f'{all_aucs_NBC[index][1]:60s} {all_aucs_NBC[index][0]:.8f} {empty:20s} {opt_aucs_NBC[index][1]:60s} {opt_aucs_NBC[index][0]:.8f}\n')


				delimiter = ','
				filename = f'results_UAI/auc/{name}_GeFs.dat'

				empty = ' '

				with open(filename, 'a') as f:
					f.write(f'\nRanking of all metrics and combinations by AUC: {empty:44s} Ranking of all metrics and OPTIMAL combinations by AUC:\n\n')
					for index in range(len(all_aucs_GeFs)):
						f.write(f'{all_aucs_GeFs[index][1]:60s} {all_aucs_GeFs[index][0]:.8f} {empty:20s} {opt_aucs_GeFs[index][1]:60s} {opt_aucs_GeFs[index][0]:.8f}\n')


		if store_results:
			name = f"{set_name}_{seed}" + train_N_suffix
			# store the mean and stddev of the AUCs for each metric across the different fixed metrics
			filename = f'results_UAI/DS_auc/{name}_NBC_summary.dat'
			with open(filename, 'a') as f:
				# f.write(f'Metric, Mean AUC, Stddev AUC\n\n')
				for metric in metrics_to_compute:
					# write metric: auc1 (stddev1), auc2 (stddev2), ...
					f.write(f'{metric:20s}: ' + ', '.join([f'{auc:.8f} ({stddev:.8f})' for auc, stddev in rob_metrics_shifted_NBC[metric]]) + '\n')

			name = f"{set_name}_{seed}" + train_N_suffix
			# store the mean and stddev of the AUCs for each metric across the different fixed metrics
			filename = f'results_UAI/DS_auc/{name}_GeFs_summary.dat'
			with open(filename, 'a') as f:
				# f.write(f'Metric, Mean AUC, Stddev AUC\n\n')
				for metric in metrics_to_compute:
					# write metric: auc1 (stddev1), auc2 (stddev2), ...
					f.write(f'{metric:20s}: ' + ', '.join([f'{auc:.8f} ({stddev:.8f})' for auc, stddev in rob_metrics_shifted_GeFs[metric]]) + '\n')


--------------- Running experiment with seed: 0 ---------------

 ------- student_port_bin -------



  0%|          | 0/7 [00:00<?, ?it/s]

(50, 29) (50,)


 14%|█▍        | 1/7 [00:13<01:19, 13.17s/it]

(50, 29) (50,)


 29%|██▊       | 2/7 [00:24<01:01, 12.32s/it]

... (135 outputs left out)

 86%|████████▌ | 6/7 [03:47<00:38, 38.03s/it]

(200, 29) (200,)


100%|██████████| 7/7 [04:25<00:00, 37.99s/it]


Accuracy NBC:  0.7628240485383343
Accuracy GeFs:  0.8157749586321015


The latex file is generated in the folder `results_UAI/latex_plots/ARC_grid/` and can be compiled using lualatex to generate the plots.

In [18]:
#### ARC grid for all metrics ####

folder_file_to_read = 'results_UAI/accaccs/'
data_set_name = 'student_port_bin'



random_number = '_0'
sizes = ['_50', '_100', '_200']
noises = ['_noise_0.0', '_noise_0.1', '_noise_0.2']

# model = '_NBC'
model = '_GeFs'

file_extension = '.dat'


uncertainty_metrics = ['margin_of_confidence', 'max_prob', 'aleatoric', 'total', 'epistemic', 'entropies']
robustness_metrics = ['global_eps', 'local_eps']

rob_metrics = {}

for size in sizes:
	for noise in noises:
		file_to_read = folder_file_to_read + data_set_name + random_number + size + noise + model + file_extension

		with open(file_to_read, 'r') as f:
			lines = f.readlines()

			store_next_line = False
			current = None
			for line in lines:
				if 'fixed' in line.strip():
					metric = line.split(':')[1].strip()
					store_next_line = True
					current = 'rob'
					continue

				if 'uncer' in line.strip():
					metric = line.split(':')[1].strip()
					store_next_line = True
					current = 'unc'
					continue

				if 'hybri' in line.strip():
					metric = line.split(':')[1].strip()
					store_next_line = True
					current = 'hyb'
					continue

				if store_next_line:
					rob_metrics[metric] = [x for x in line.strip().split(',')]

					current = None
					store_next_line = False


		folder_file_to_write = 'results_UAI/latex_plots/ARC_grid/files/'

		filename = folder_file_to_write + data_set_name + size + noise + model + '_arc.csv'

		metrics = uncertainty_metrics + robustness_metrics

		with open(filename, 'w') as f:
			f.write('n'+','+','.join(metrics)+'\n')

			for i in range(len(rob_metrics['global_eps'])):
				line = str(i)
				for metric in metrics:
					line += ',' + rob_metrics[metric][i]
				f.write(line+'\n')

In [11]:
#### AUC TABLE ####
folder_file_to_read = 'results_UAI/auc/'

data_set_names = ['adult',
					'Australian',
					'bank']

# data_set_names = ['adult',
# 					'Australian',
# 					'bank',
# 					'BCW',
# 					'german',
# 					'heart_disease',
# 					'lymphography',
# 					'Healthy_aging',
# 					'nursery',
# 					'solar_big',
# 					'solar_small',
# 					'spect_heart',
# 					'student_math_bin',
# 					'student_port_bin']

random_number = '_42'
Ndata_suffix = '_50'

# noise_factor = 0.2
noise_factor = 0.0

if noise_factor > 0:
	noise = True
else:
	noise = False

noise_suffix = f'_noise_{noise_factor}'

nbc_model = False

if nbc_model:
	model_architecture = '_NBC'
else:
	model_architecture = '_GeFs'


file_extension = '.dat'

data_set_dicts = {}

uncertainty_metrics = ['margi', 'max_p', 'aleat', 'total', 'epist', 'entro']

precision = 5

for data_set_name in data_set_names:
	file_to_read = folder_file_to_read + data_set_name + random_number + Ndata_suffix + noise_suffix + model_architecture + file_extension

	auc_metrics = {}

	with open(file_to_read, 'r') as f:
		lines = f.readlines()

		auc_metrics['global'] = float(lines[0].split(':')[1].strip()[:precision+2])
		auc_metrics['local'] = float(lines[2 * len(uncertainty_metrics) + 3].split(':')[1].strip()[:precision+2])

		for i in range(len(uncertainty_metrics)):
			auc_metrics[uncertainty_metrics[i]] = float(lines[i*2].split(':')[1].strip().split(' ')[1][:precision+2])

		for i in range(2*len(uncertainty_metrics)):
			rob_metric = 'global' if i < len(uncertainty_metrics) else 'local'
			uncertainty_metric = uncertainty_metrics[i % len(uncertainty_metrics)]

			if i < len(uncertainty_metrics):
				auc = float(lines[i*2].split(':')[1].strip().split(' ')[2][:precision+2])
				optimal_auc = float(lines[i*2].split(':')[1].strip().split(' ')[3][1:precision+3])
				mixing = lines[i*2].split('Hybrid')[1].strip()[:4]
				optimal_mixing = lines[i*2].split('Hybrid')[1].strip()[6:10]
			else:
				auc = float(lines[i*2+1].split(':')[1].strip().split(' ')[2][:precision+2])
				optimal_auc = float(lines[i*2+1].split(':')[1].strip().split(' ')[3][1:precision+3])
				mixing = lines[i*2+1].split('Hybrid')[1].strip()[:4]
				optimal_mixing = lines[i*2+1].split('Hybrid')[1].strip()[6:10]
			auc_metrics[f'{rob_metric}+{uncertainty_metric}'] = (auc, optimal_auc, mixing, optimal_mixing)

	data_set_dicts[data_set_name] = auc_metrics

folder_file_to_write = 'results_UAI/latex_plots/AUCs/files/'


uncertainty_metrics = ['margin', 'maxprob', 'aleatoric', 'total', 'epistemic', 'entropies']
robustness_metrics = ['robglobal', 'roblocal']

metrics = uncertainty_metrics + robustness_metrics

file_to_write = folder_file_to_write + 'table_all_metrics' + random_number + model_architecture
if noise:
	file_to_write += '_DS'
file_to_write += '.csv'

with open(file_to_write, 'w') as f:
	header = ['dataset'] + metrics
	# write header line with all metrics separated by commas
	f.write(','.join(header)+'\n')

	for i, data_set_name in enumerate(data_set_names):

		dsn = 'D' + f"{i+1}"

		unc_margin = data_set_dicts[data_set_name]['margi']
		unc_maxprob = data_set_dicts[data_set_name]['max_p']
		unc_aleatoric = data_set_dicts[data_set_name]['aleat']
		unc_total = data_set_dicts[data_set_name]['total']
		unc_epistemic = data_set_dicts[data_set_name]['epist']
		unc_entropy = data_set_dicts[data_set_name]['entro']

		rob_global = data_set_dicts[data_set_name]['global']
		rob_local = data_set_dicts[data_set_name]['local']

		dsn = dsn[0].upper() + dsn[1:]

		f.write(dsn.replace('_',' ') + ',' +
					f'{unc_maxprob:.4f}' + ',' +
					f'{unc_margin:.4f}' + ',' +
					f'{unc_entropy:.4f}' + ',' +
					f'{unc_total:.4f}' + ',' +
					f'{unc_aleatoric:.4f}' + ','
					f'{unc_epistemic:.4f}' + ',' +
					f'{rob_global:.4f}' + ',' +
					f'{rob_local:.4f}'
					+'\n')